In [17]:
from sympy import *
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [18]:
g=Symp_symb(7)
C=g.cochain_complex
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis
P=RegularCartanGeometry(g,'eta')
eta=IndexedBase('eta')
P.fund_invars=[eta[3,9,6],eta[6,9,9],eta[3,9,4]]

In [19]:
def der_term(P,i0,i1,i2):
    """Returns the derivative term for the component of the Bianchi identity
    applied to (i0,i1,i2), which is an element of P.symbol."""
    g=P.symbol
    X0,X1,X2=[g.basis[a] for a in [i0,i1,i2]]
    w0,w1,w2=[None,None,None]
    try: w0=g.ext_alg.elt_from_cd({(str(X1),str(X2)):1})
    except(KeyError): pass
    try: w1=g.ext_alg.elt_from_cd({(str(X2),str(X0)):1})
    except(KeyError): pass
    try: w2=g.ext_alg.elt_from_cd({(str(X0),str(X1)):1})
    except(KeyError): pass

    r=g.elt()
    if w0!=None: r+=P.fund_der(P.curvature.apply_cochain_map(w0),i0)
    if w1!=None: r+=P.fund_der(P.curvature.apply_cochain_map(w1),i1)
    if w2!=None: r+=P.fund_der(P.curvature.apply_cochain_map(w2),i2)
    return r

def cb_term(P,i0,i1,i2):
    """returns the coboundary of P.curvature applied to i0,i1,i2 elements of P.symbol.basis.
    This is needed because we care about the value as a cochain in C(g,g), not just C(m,g)
    (at least for the purpose of checks)"""

    r=P.symbol.elt()
    X0,X1,X2=[P.symbol.basis[a] for a in [i0,i1,i2]]
    
    for i in range(3):
        Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
        w0=Y0.negative_projection().cast_as_ext_elt().wedge(Y1.negative_projection().cast_as_ext_elt())
        r+=P.curvature.apply_cochain_map(w0).ad(Y2)
        w1=Y0.ad(Y1).negative_projection().cast_as_ext_elt().wedge(Y2.negative_projection().cast_as_ext_elt())
        r+=P.curvature.apply_cochain_map(w1)
    return -r

# What if instead of computing these individually, I just computed Gerstenhaber square
# of curvature once, then 
def gerst_term(P,i0,i1,i2):
    X0,X1,X2=[P.symbol.basis[a] for a in [i0,i1,i2]]
    r=P.symbol.elt()
    for i in range(3):
        Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
        Y0=Y0.negative_projection().cast_as_ext_elt()
        Y1=Y1.negative_projection().cast_as_ext_elt()
        Y2=Y2.negative_projection().cast_as_ext_elt()
        w=P.curvature.apply_cochain_map(Y0.wedge(Y1)).negative_projection().cast_as_ext_elt()
        r+=P.curvature.apply_cochain_map(w.wedge(Y2))
    return r

def Bianchi(P,i0,i1,i2,subdivide=False):
    r0=cb_term(P,i0,i1,i2)
    r1=der_term(P,i0,i1,i2)
    r2=gerst_term(P,i0,i1,i2)
    if subdivide: return (r0, r1, r2)
    return r0+r1+r2
    # if subdivide: return (cb_term(P,i0,i1,i2), der_term(P,i0,i1,i2), gerst_term(P,i0,i1,i2))
    # return cb_term(P,i0,i1,i2)+der_term(P,i0,i1,i2)+gerst_term(P,i0,i1,i2)

### Computations

In [20]:
D=Distr_of_constant_symbol(g,-P.curvature)

In [ ]:
tuples_by_wght={}
for i in range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        for k in range(j+1,len(g.basis)):
            for m in range(len(g.basis)):
                w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[m].wght
                if w not in tuples_by_wght: tuples_by_wght[w]=[]
                tuples_by_wght[w].append((i,j,k,m))

for w in range(min(list(tuples_by_wght.keys())),max(list(tuples_by_wght.keys()))+1):
    print(w,'-->',len(tuples_by_wght[w]))

In [28]:
Bianchi_dict={}

def choose_var(expr,symb,excl_list=[]):
    """Returns a variable of maximal weight from the expression, which should be a rational
    expression in 2-tensors"""
    s=Indexed_obj_in_expr(expr)
    r=next(iter(s))
    for a in s:
        if (wght_of_ind(a,symb)<wght_of_ind(r,symb) and
                r.base[r.indices[0:3]] not in excl_list): 
            r=a
    if r.base[r.indices[0:3]] in excl_list: return None
    return r

def compute_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        time0=time.time()
        print('Computing', t)
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else: 
            time1=time.time()
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
            print('    Bianchi computed in',hrs_min_sec(time.time()-time1))
        time2=time.time()
        to_solve=ds_subs(zero_elt.vec[m],Bianchi_dict,D)
        print('    to_solve computed in',hrs_min_sec(time.time()-time2))
        if simplify(to_solve)!=0:
            time3=time.time()
            s=find_a_linear_term(to_solve,P.fund_invars)
            if s==None: s=choose_var(to_solve,g,P.fund_invars)
            if s==None: s=choose_var(to_solve,g)
            sol=solve(to_solve,s,dict=True)[0]
            print('    Solving complete in',hrs_min_sec(time.time()-time3))
            time4=time.time()
            for a in sol: 
                ds_add_key(a,sol[a],Bianchi_dict,D)
            print('    Substitution complete in',hrs_min_sec(time.time()-time4))
        print('   ',t,'computed in',hrs_min_sec(time.time()-time0))
        # Notice that D.curv = -P.curvature, since I switched sign conventions

def check_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else:
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
        r=simplify(ds_subs(zero_elt.vec[m],Bianchi_dict,D))
        if r!=0: print(r)

In [ ]:
for w in range(1,20):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [ ]:
for w in range(1,20):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [ ]:
for w in range(1,20):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [31]:
# with shelve.open('Abstract_Syzygies') as shelf:
#     shelf['Bianchi_dict'+'{j}'.format(j=7)]=Bianchi_dict

In [30]:
saved_Bianchi_dict=copy.deepcopy(Bianchi_dict) # Through wght 14 at the moment
saved_curv=copy.deepcopy(P.curvature)

In [60]:
with shelve.open('Abstract_Syzygies') as shelf:
    temp=shelf['Bianchi_dict'+'{j}'.format(j=7)]

In [ ]:
for a in P.fund_invars:
    print(a, a in Bianchi_dict)

In [ ]:
Bianchi_dict[P.fund_invars[1]]

In [32]:
def hrs_min_sec(sec_val):
    hours=str(round(sec_val//(60**2)))
    minutes=str(round((sec_val//60)%60))
    seconds=str(round(sec_val%60,0))
    if sec_val//(60**2)!=0:
        return(hours+' hrs '+minutes+' min '+seconds+' sec')
    if (sec_val//60)%60!=0:
        return(minutes+' min '+seconds+' sec')
    return(seconds+' sec')

In [51]:
# Bianchi_dict=copy.deepcopy(saved_Bianchi_dict) # Through wght 13 at the moment
# P.curvature=copy.deepcopy(saved_curv)

In [45]:
P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
D.curv=-P.curvature

In [ ]:
for w in range(1,13):
    print(w)
    check_Bianchi(w)

In [ ]:
# These should only be derivatives of the fundamental invariants
temp=set()
for k1 in Bianchi_dict:
    for k2 in Bianchi_dict[k1]:
        temp=temp.union(Indexed_obj_in_expr(Bianchi_dict[k1][k2]))
temp

In [49]:
syzygies_by_wght={}
for k in P.fund_invars:
    for j in Bianchi_dict[k]:
        w=-wght_of_ind(k.base[k.indices+j],g)
        if not w in syzygies_by_wght: syzygies_by_wght[w]=[]
        new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
        if new_syzygy!=0:
            new_syzygy=new_syzygy*new_syzygy.as_numer_denom()[1]
            syzygies_by_wght[w].append(new_syzygy)

In [ ]:
for a in syzygies_by_wght[9]:
    display(a)

In [ ]:
A1,A2,A3,A4,A5=syzygies_by_wght[8]


B1=(A2*lcm(30,465)/30-A3*lcm(30,465)/465)
B2=(A2*lcm(30,1305)/30-A4*lcm(30,1305)/1305)
display(A1)
display(A2)
display(A3)
display(A4)
display(A5)
# display(4*factor(B1/7))
# display(factor(B2/7))
# display(factor(4*B1-B2))

In [ ]:
A1=syzygies_by_wght[8][2]
A2=syzygies_by_wght[8][3]
A3=syzygies_by_wght[8][4]

B1=(A1*lcm(30,465)/30-A2*lcm(30,465)/465)
B2=(A1*lcm(30,1305)/30-A3*lcm(30,1305)/1305)
display(A1)
display(A2)
display(A3)
# display(4*factor(B1/7))
# display(factor(B2/7))
# display(factor(4*B1-B2))

In [58]:
I=IndexedBase('I')
W=IndexedBase('W')

fund_invar_dict={I[3]:eta[6,9,9],W[4]:eta[3,9,6],W[6]:eta[3,9,4]}

def convert_syzygy(syz):
    T=Indexed_obj_in_expr(syz)
    s={}
    for t in T:
        temp=fund_invar_dict[t.base[t.indices[0]]]
        s[t]=temp.base[temp.indices+t.indices[1:len(t.indices)]]
    return syz.xreplace(s)

In [ ]:
old_syzygies=[
    7*I[3, 3, 3] + W[4, 4],
    14*I[3, 3, 3, 3, 4] - 7*I[3, 3, 3, 4, 3] + W[4, 3, 4, 4],
    -7*I[3, 3, 3, 3, 4, 4] + 14*I[3, 3, 3, 4, 3, 4] - 7*I[3, 3, 3, 4, 4, 3],
    -7*I[3, 3, 3, 3, 4, 4] + 14*I[3, 3, 3, 4, 3, 4] - 7*I[3, 3, 3, 4, 4, 3],
    -28*I[3, 3, 3, 3, 3, 3]/5 - 96*I[3, 3]*W[4]/5 - 12*I[3]*W[4, 3]/5 - 7*W[4, 3, 3, 3, 4]/30 + W[4, 3, 3, 4, 3] - 3*W[4, 3, 4, 3, 3]/2 + 49*W[6, 3, 4]/1650 - 14*W[6, 4, 3]/275
]

for old_syz in old_syzygies:
    print(simplify(ds_subs(convert_syzygy(old_syz),Bianchi_dict,D)))

In [ ]:
C.subspace_proj(ds_subs(D.curv.wght_proj(4),Bianchi_dict,D),'harmonic')
C.subspace_proj(ds_subs(D.curv.wght_proj(4),Bianchi_dict,D),'coexact')

In [ ]:
ds_subs(P.fund_der(eta[6,9,9],3),Bianchi_dict,D)

In [29]:
partial_Bianchi_dicts={20:copy.deepcopy(Bianchi_dict)}
for w in reversed(range(20)):
    r=copy.deepcopy(partial_Bianchi_dicts[w+1])
    for k in r:
        for j in list(r[k].keys()):
            if -wght_of_ind(k.base[k.indices+j],g)>w: r[k].pop(j)
    for k in list(r.keys()):
        if r[k]==dict(): r.pop(k)
    partial_Bianchi_dicts[w]=r

In [ ]:
harm_basis={}
for i in [3,4,6]:
    harm_basis[i]=C.elt({})
    for j in range(len(C.basis(2,i))):
        harm_basis[i]=harm_basis[i]+C.subspace_basis('harmonic',2,i)[j]*C.basis(2,i)[j]
